# Notebook 06: Interactive Portfolio Dashboard

**Project:** GNN-Based Battery Voltage Predictor  

---

## Overview

This notebook assembles all results into a standalone interactive HTML dashboard
for portfolio presentation. The dashboard contains four panels:

1. **Parity plot**: Predicted vs actual voltage (all models, colored by chemistry)
2. **Model comparison**: MAE and RMSE bar chart for RF, CGCNN, M3GNet
3. **Screening map**: Predicted voltage vs estimated capacity for novel candidates,
   with hover labels showing formula and stability
4. **Candidate table**: Top 50 screened candidates with sortable columns

All panels use Plotly for interactivity. The final figure is exported to
`results/dashboard.html` as a self-contained file (no server required).

In [ ]:
import sys
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.evaluate import PALETTE, compute_metrics

DATA_DIR    = project_root / 'data'
MODELS_DIR  = project_root / 'models'
RESULTS_DIR = project_root / 'results'

print('Plotly dashboard builder ready.')

## 1. Load Saved Results

We load predictions saved by Notebook 04 (evaluation) and the top candidates
from Notebook 05 (screening). If running for the first time, run notebooks
04 and 05 first to generate these files.

In [ ]:
# Load predictions from evaluation results
import torch
from src.data import VoltageGraphDataset
from src.models import build_cgcnn
from src.train import predict, make_loaders
from src.utils import set_seed

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load splits
with open(DATA_DIR / 'splits.pkl', 'rb') as f:
    splits = pickle.load(f)
test_entries = splits['test']

# Load matminer features
with open(DATA_DIR / 'matminer_filtered.pkl', 'rb') as f:
    mm = pickle.load(f)
X_test, y_test_mm = mm['X_test'], mm['y_test']

# Load RF model
with open(MODELS_DIR / 'rf_model.pkl', 'rb') as f:
    rf_model = pickle.load(f)

# Load test graphs
test_ds = VoltageGraphDataset.from_processed(str(DATA_DIR / 'test_graphs.pkl'))
_, _, test_loader = make_loaders(test_ds, test_ds, test_ds, batch_size=128, num_workers=0)

# Load CGCNN
cgcnn = build_cgcnn(node_dim=9, edge_dim=64, hidden_dim=128, n_conv=4, dropout=0.1)
cgcnn.load_state_dict(torch.load(MODELS_DIR / 'cgcnn_best.pt', map_location=device))
cgcnn = cgcnn.to(device)

cgcnn_pred, cgcnn_true = predict(cgcnn, test_loader, device)
test_chemistry = [e['chemistry_family'] for e in test_entries
                  if e.get('charged_structure') or e.get('discharged_structure')]
n_test = len(cgcnn_pred)

# RF predictions
rf_pred = rf_model.predict(X_test[:n_test])
y_test_rf = y_test_mm[:n_test]

# Compute metrics
all_metrics = {
    'Random Forest': compute_metrics(y_test_rf, rf_pred),
    'CGCNN':         compute_metrics(cgcnn_true, cgcnn_pred),
}

# Load top candidates
top_candidates = pd.read_csv(RESULTS_DIR / 'top_candidates.csv')
df_screen_all = pd.read_csv(RESULTS_DIR / 'top_candidates.csv')  # reuse top50

print(f'Test set: {n_test} entries')
print(f'Top candidates: {len(top_candidates)}')
print('Results loaded.')

In [ ]:
# Load M3GNet if model file exists
try:
    import matgl
    import dgl
    import torch.nn as nn
    from matgl.ext.pymatgen import Structure2Graph, get_element_list
    from pymatgen.core import Structure
    from torch.utils.data import DataLoader as TorchDataLoader

    class M3GNetRegressor(nn.Module):
        def __init__(self, backbone, hidden_dim=64):
            super().__init__()
            self.backbone = backbone
            backbone_dim = getattr(backbone.model, 'dim_node_embedding', 64)
            self.head = nn.Sequential(
                nn.Linear(backbone_dim, hidden_dim), nn.SiLU(), nn.Linear(hidden_dim, 1),
            )
        def forward(self, g, lat, state):
            node_feats = self.backbone.model.get_graph_features(g, lat, state)
            pooled = dgl.mean_nodes(g, feat='h') if 'h' in g.ndata else node_feats.mean(0)
            return self.head(pooled).squeeze(-1)

    pretrained_m3g = matgl.load_model('M3GNet-MP-2021.2.8-PES')
    m3gnet_model = M3GNetRegressor(pretrained_m3g).to(device)
    m3gnet_model.load_state_dict(torch.load(MODELS_DIR / 'm3gnet_best.pt', map_location=device))
    m3gnet_model.eval()

    # Quick test set predictions
    all_structs = []
    for e in splits['train'] + splits['val'] + splits['test']:
        sd = e.get('charged_structure') or e.get('discharged_structure')
        if sd:
            try: all_structs.append(Structure.from_dict(sd))
            except: pass

    element_types = get_element_list(all_structs)
    converter = Structure2Graph(element_types=element_types, cutoff=5.0)

    class M3GDataset(torch.utils.data.Dataset):
        def __init__(self, entries, converter):
            self.graphs, self.targets = [], []
            for e in entries:
                sd = e.get('charged_structure') or e.get('discharged_structure')
                if sd is None: continue
                try:
                    s = Structure.from_dict(sd)
                    g, lat, state = converter.get_graph(s)
                    self.graphs.append((g, lat, state))
                    self.targets.append(e['average_voltage'])
                except: pass
        def __len__(self): return len(self.graphs)
        def __getitem__(self, i): return self.graphs[i], torch.tensor(self.targets[i], dtype=torch.float32)

    def collate_fn(batch):
        graphs, targets = zip(*batch)
        batched = dgl.batch([g[0] for g in graphs])
        lat = torch.stack([g[1] for g in graphs])
        state_list = [g[2] for g in graphs]
        state = torch.stack(state_list) if state_list[0] is not None else None
        return (batched, lat, state), torch.stack(targets)

    m3g_test_ds = M3GDataset(test_entries, converter)
    m3g_loader = TorchDataLoader(m3g_test_ds, batch_size=32, collate_fn=collate_fn)

    m3g_preds, m3g_true = [], []
    with torch.no_grad():
        for (g, lat, state), targets in m3g_loader:
            g = g.to(device)
            lat = lat.to(device)
            if state is not None: state = state.to(device)
            targets = targets.to(device)
            preds = m3gnet_model(g, lat, state)
            m3g_preds.extend(preds.cpu().numpy())
            m3g_true.extend(targets.cpu().numpy())

    m3g_pred_arr = np.array(m3g_preds)
    m3g_true_arr = np.array(m3g_true)
    all_metrics['M3GNet'] = compute_metrics(m3g_true_arr, m3g_pred_arr)
    print('M3GNet predictions loaded.')
except Exception as e:
    print(f'M3GNet not available (skipping): {e}')
    m3g_pred_arr, m3g_true_arr = None, None

## 2. Build Interactive Dashboard

The dashboard uses Plotly subplots arranged in a 2x2 grid:

- Top left: Parity plot (predicted vs actual) for best model, colored by chemistry
- Top right: Model comparison bar chart (MAE and RMSE)
- Bottom left: Screening scatter (voltage vs capacity for novel candidates)
- Bottom right: Top 20 candidate table

In [ ]:
# Use CGCNN predictions for the parity plot (or M3GNet if available)
if m3g_pred_arr is not None:
    parity_true = m3g_true_arr
    parity_pred = m3g_pred_arr
    parity_chem = test_chemistry[:len(m3g_pred_arr)]
    parity_model_name = 'M3GNet'
else:
    parity_true = cgcnn_true
    parity_pred = cgcnn_pred
    parity_chem = test_chemistry[:len(cgcnn_pred)]
    parity_model_name = 'CGCNN'

# Map chemistry families to Plotly-friendly hex colors
PLOT_COLORS = {
    'oxide': '#2196F3', 'phosphate': '#4CAF50',
    'sulfide': '#FF9800', 'fluoride': '#9C27B0',
    'sulfate': '#F44336', 'silicate': '#009688', 'other': '#607D8B',
}

# Build parity figure
parity_fig_data = []
for fam in sorted(set(parity_chem)):
    mask = np.array(parity_chem) == fam
    parity_fig_data.append(go.Scatter(
        x=parity_true[mask].tolist(),
        y=parity_pred[mask].tolist(),
        mode='markers',
        name=fam,
        marker=dict(color=PLOT_COLORS.get(fam, '#607D8B'), size=5, opacity=0.6),
    ))

met = all_metrics.get(parity_model_name, compute_metrics(parity_true, parity_pred))
vmin = float(min(parity_true.min(), parity_pred.min())) - 0.2
vmax = float(max(parity_true.max(), parity_pred.max())) + 0.2
parity_fig_data.append(go.Scatter(
    x=[vmin, vmax], y=[vmin, vmax],
    mode='lines', showlegend=False,
    line=dict(color='black', dash='dash', width=1),
))

# Build model comparison bar chart
model_names = list(all_metrics.keys())
maes = [all_metrics[m]['MAE'] for m in model_names]
rmses = [all_metrics[m]['RMSE'] for m in model_names]
model_colors = ['#E91E63', '#2196F3', '#4CAF50']

# Build screening scatter
screen_fig_data = []
for fam in top_candidates['chemistry_family'].unique():
    subset = top_candidates[top_candidates['chemistry_family'] == fam]
    screen_fig_data.append(go.Scatter(
        x=subset['est_capacity_mAh_g'].tolist(),
        y=subset['predicted_voltage_V'].tolist(),
        mode='markers',
        name=fam,
        marker=dict(color=PLOT_COLORS.get(fam, '#607D8B'), size=8, opacity=0.8),
        text=subset['formula'].tolist(),
        hovertemplate=(
            '<b>%{text}</b><br>'
            'Predicted: %{y:.3f} V<br>'
            'Est. Capacity: %{x:.1f} mAh/g<extra></extra>'
        ),
    ))

print('Data prepared for dashboard.')

In [ ]:
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        f'{parity_model_name}: Predicted vs Actual Voltage (Test Set)',
        'Model Comparison: MAE and RMSE',
        'Screening: Novel Li Candidates (Predicted Voltage vs Capacity)',
        'Top 20 Screened Candidates',
    ],
    specs=[
        [{'type': 'scatter'}, {'type': 'bar'}],
        [{'type': 'scatter'}, {'type': 'table'}],
    ],
    column_widths=[0.55, 0.45],
    row_heights=[0.5, 0.5],
    vertical_spacing=0.12,
    horizontal_spacing=0.10,
)

# Panel 1: Parity plot
for trace in parity_fig_data:
    fig.add_trace(trace, row=1, col=1)

# Panel 2: Model comparison bars
for i, (mn, mae, rmse, color) in enumerate(zip(model_names, maes, rmses, model_colors[:len(model_names)])):
    fig.add_trace(go.Bar(
        name=mn, x=['MAE', 'RMSE'],
        y=[mae, rmse],
        marker_color=color,
        opacity=0.85,
        text=[f'{mae:.3f}', f'{rmse:.3f}'],
        textposition='outside',
        showlegend=False,
    ), row=1, col=2)

# Panel 3: Screening scatter
for trace in screen_fig_data:
    trace.showlegend = False
    fig.add_trace(trace, row=2, col=1)

# Add 2.5 V cutoff line
fig.add_hline(y=2.5, line_dash='dot', line_color='grey', opacity=0.5, row=2, col=1)

# Panel 4: Top 20 candidates table
top20 = top_candidates.head(20)
fig.add_trace(go.Table(
    header=dict(
        values=['Rank', 'Formula', 'Family', 'Pred. V (V)', 'Capacity (mAh/g)', 'Ehull (meV/atom)'],
        fill_color='#1565C0',
        font=dict(color='white', size=11),
        align='center',
    ),
    cells=dict(
        values=[
            top20['rank'].tolist(),
            top20['formula'].tolist(),
            top20['chemistry_family'].tolist(),
            top20['predicted_voltage_V'].round(3).tolist(),
            top20['est_capacity_mAh_g'].round(1).tolist(),
            top20['energy_above_hull_meV'].round(1).tolist(),
        ],
        fill_color=[['#E3F2FD' if i % 2 == 0 else 'white' for i in range(len(top20))]],
        align='center',
        font=dict(size=10),
        height=22,
    ),
), row=2, col=2)

# Axis labels
fig.update_xaxes(title_text='DFT Voltage (V vs Li/Li+)', row=1, col=1)
fig.update_yaxes(title_text='Predicted Voltage (V)', row=1, col=1)
fig.update_yaxes(title_text='Error (V)', row=1, col=2)
fig.update_xaxes(title_text='Estimated Capacity (mAh/g)', row=2, col=1)
fig.update_yaxes(title_text='Predicted Voltage (V vs Li/Li+)', row=2, col=1)

# Add parity line annotation
fig.update_layout(
    height=900,
    width=1300,
    title=dict(
        text='GNN-Based Battery Voltage Predictor: Portfolio Dashboard',
        font=dict(size=16),
        x=0.5,
    ),
    font=dict(family='Arial, Helvetica, sans-serif', size=11),
    paper_bgcolor='white',
    plot_bgcolor='#FAFAFA',
    legend=dict(
        title='Chemistry Family',
        orientation='v',
        x=1.01, y=0.95,
        bgcolor='rgba(255,255,255,0.8)',
        bordercolor='#CCCCCC',
        borderwidth=1,
    ),
    barmode='group',
    margin=dict(t=80, b=40, l=50, r=160),
)

# Annotation with metrics in parity panel
met_text = (
    f"MAE = {met['MAE']:.3f} V | "
    f"RMSE = {met['RMSE']:.3f} V | "
    f"R² = {met['R2']:.3f}"
)
fig.add_annotation(
    text=met_text, xref='paper', yref='paper',
    x=0.01, y=0.97, showarrow=False,
    font=dict(size=10, color='#333333'),
    bgcolor='rgba(255,255,255,0.8)',
    bordercolor='grey', borderwidth=1,
)

print('Dashboard assembled.')
fig.show()

In [ ]:
output_path = RESULTS_DIR / 'dashboard.html'
fig.write_html(
    str(output_path),
    include_plotlyjs='cdn',
    full_html=True,
    config={'displayModeBar': True, 'scrollZoom': True},
)
print(f'Dashboard exported to {output_path}')
print(f'File size: {output_path.stat().st_size / 1024:.1f} KB')
print(f'\nOpen in browser: open {output_path}')

## 3. Additional Figures for Portfolio Page

In [ ]:
# Interactive voltage distribution colored by chemistry (bonus figure)
import json as _json

with open(DATA_DIR / 'battery_dataset.json') as f:
    all_entries = _json.load(f)

df_all = pd.DataFrame([{
    'average_voltage': e['average_voltage'],
    'chemistry_family': e['chemistry_family'],
    'formula': e['formula'],
    'capacity_grav': e['capacity_grav'],
} for e in all_entries])

fig_dist = px.histogram(
    df_all, x='average_voltage',
    color='chemistry_family',
    color_discrete_map=PLOT_COLORS,
    nbins=60,
    title='Li Intercalation Voltage Distribution by Chemistry Family',
    labels={'average_voltage': 'Average Voltage (V vs Li/Li+)', 'count': 'Number of Entries'},
    barmode='overlay',
    opacity=0.75,
    template='plotly_white',
)
fig_dist.update_layout(
    font=dict(family='Arial, Helvetica, sans-serif'),
    legend_title='Chemistry Family',
)
fig_dist.write_html(str(RESULTS_DIR / 'fig_voltage_distribution_interactive.html'),
                    include_plotlyjs='cdn')
fig_dist.show()
print('Saved interactive voltage distribution.')

In [ ]:
print('Dashboard notebook complete!')
print(f'Outputs in {RESULTS_DIR}:')
for p in sorted(RESULTS_DIR.glob('*.*')):
    print(f'  {p.name:45s} {p.stat().st_size / 1024:.1f} KB')